In [1]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer


In [2]:
from datasets import load_dataset

dataset = load_dataset("Salesforce/wikitext", "wikitext-2-v1")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


In [3]:
print(dataset)

DatasetDict({
    test: Dataset({
        features: ['text'],
        num_rows: 4358
    })
    train: Dataset({
        features: ['text'],
        num_rows: 36718
    })
    validation: Dataset({
        features: ['text'],
        num_rows: 3760
    })
})


In [4]:
df = pd.DataFrame(dataset['train'])

In [5]:
df.head()

,text
0,
1,= Valkyria Chronicles III = \n
2,
3,Senjō no Valkyria 3 : <unk> Chronicles ( Japa...
4,"The game began development in 2010 , carrying..."


In [6]:
df.isnull().sum()

,0
text,0


In [ ]:
print(f"Total rows: {len(df)}")

Total rows: 36718


In [8]:
df = df[df['text'].str.strip() != '']

In [ ]:
print(f"Total rows: {len(df)}")

Total rows: 23767


In [10]:
tokenizer = Tokenizer()

In [11]:
tokenizer.fit_on_texts(df['text'])

In [12]:
len(tokenizer.word_index)

28780

In [13]:
df['tokenized'] = tokenizer.texts_to_sequences(df['text'])
print(df[['text', 'tokenized']].head(10))

                                                 text  \
1                      = Valkyria Chronicles III = \n   
3    Senjō no Valkyria 3 : <unk> Chronicles ( Japa...   
4    The game began development in 2010 , carrying...   
5    It met with positive sales in Japan , and was...   
7                                 = = Gameplay = = \n   
9    As with previous <unk> Chronicles games , Val...   
10   The game 's battle system , the <unk> system ...   
11   Troops are divided into five classes : Scouts...   
13                                    = = Plot = = \n   
15   The game takes place during the Second Europa...   

                                            tokenized  
1                                   [3779, 3842, 867]  
3   [18139, 77, 3779, 82, 3, 3842, 767, 23654, 608...  
4   [1, 61, 128, 358, 5, 286, 3220, 59, 7, 176, 17...  
5   [17, 777, 14, 917, 1426, 5, 962, 4, 8, 721, 15...  
7                                              [2073]  
9   [10, 14, 466, 3, 3842, 185, 3779

In [14]:
# input_sequences = []
# for i in range (1,len(df["tokenized"])):
#   input_sequences.append( df["tokenized"][:i+1])

input_sequences = []

for tokens in df["tokenized"]:
    for i in range(1, len(tokens)):
        input_sequences.append(tokens[:i+1])


In [15]:
max_len = max([len(x) for x in input_sequences])

In [ ]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [17]:
padded_input_sequences = pad_sequences(input_sequences,maxlen = max_len, padding = 'pre')

In [ ]:
padded_input_sequences

array([[    0,     0,     0, ...,     0,  3779,  3842],
       [    0,     0,     0, ...,  3779,  3842,   867],
       [    0,     0,     0, ...,     0, 18139,    77],
       ...,
       [    0,     0,     0, ...,   406,    10,    23],
       [    0,     0,     0, ...,    10,    23,  2435],
       [    0,     0,     0, ...,    23,  2435,  4711]], dtype=int32)

In [19]:
padded_input_sequences[:,:-1]

array([[    0,     0,     0, ...,     0,     0,  3779],
       [    0,     0,     0, ...,     0,  3779,  3842],
       [    0,     0,     0, ...,     0,     0, 18139],
       ...,
       [    0,     0,     0, ...,    26,   406,    10],
       [    0,     0,     0, ...,   406,    10,    23],
       [    0,     0,     0, ...,    10,    23,  2435]], dtype=int32)

In [ ]:
X = padded_input_sequences[:,:-1]


In [21]:
y = padded_input_sequences[:,-1]

In [22]:
X.shape

(1736132, 631)

In [ ]:
y.shape

(1736132,)

In [ ]:
from tensorflow.keras.utils import to_categorical

: 

In [25]:
y = to_categorical(y,num_classes=28781)

: 

: 

In [ ]:
y.shape

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense

In [ ]:
model = Sequential()
model.add(Embedding(28781, 100, input_length=max_len-1))
model.add(LSTM(150))
model.add(Dense(28781, activation='softmax'))


In [ ]:
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

In [ ]:
model.fit(X,y,epochs=100)